# C6-pytorch — Practice p12 — Solution


**(a) Build the polynomial layer by layer.**  The four dense counts are
$h(3+1)=4h$, $h(h+1)=h^2+h$, another
$h(h+1)=h^2+h$, and $1(h+1)=h+1$.  Thus
$$P(h)=4h+(h^2+h)+(h^2+h)+(h+1)=2h^2+7h+1.$$
The intervening gates register no parameters.

**(b) Uniqueness.**  $P'(h)=4h+7$, which is positive for every
$h>0$.  Hence $P$ is strictly increasing on the admissible widths and
cannot take one budget value at two different positive widths.

**(c) Solve the budget.**  Setting $P(h)=226$ gives
$2h^2+7h-225=0$.  The discriminant is
$7^2-4(2)(-225)=1849=43^2$, so
$$h=\frac{-7\pm43}{4}\in\{9,-25/2\}.$$
A width must be a positive integer, so $-25/2$ is inadmissible and the
unique width is $h=9$.


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)

class DenseLayer(nn.Module):
    """Session 2's pinned dense layer: weight (out, in), bias (out,)."""

    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias


class ThresholdGate(nn.Module):
    """Session 2's gate: 1 where x >= 0, else 0; owns no parameters."""

    def forward(self, x):
        return (x >= 0).to(x.dtype)


h_solved = 9
layers = (
    DenseLayer(torch.zeros(h_solved, 3), torch.zeros(h_solved)),
    DenseLayer(torch.zeros(h_solved, h_solved), torch.zeros(h_solved)),
    DenseLayer(torch.zeros(h_solved, h_solved), torch.zeros(h_solved)),
    DenseLayer(torch.zeros(1, h_solved), torch.zeros(1)),
)
torch_count = sum(p.numel() for layer in layers for p in layer.parameters())
poly_check = 2 * h_solved ** 2 + 7 * h_solved + 1
anchor_ok = (torch_count == 226) and (poly_check == 226)

assert anchor_ok
h_solved, torch_count, poly_check, anchor_ok


### Answer check


In [ ]:
assert h_solved == 9
assert torch_count == poly_check == 226
assert 2 * (-25 / 2) ** 2 + 7 * (-25 / 2) + 1 == 226
assert -25 / 2 < 1
